# Spatially-Aware Composite Lateral Variability Indices
## Area A vs Area B — five petrophysical properties, four spatial indices

The original WHI, PTSI, and JES treated every grid node as an independent observation,
ignoring *where* it sits in space. This notebook replaces them with spatially-aware
versions that explicitly incorporate x/y coordinates and inter-point distances.

| Original | Spatial upgrade | Key spatial ingredient |
|----------|----------------|----------------------|
| WHI | **SW-WHI** — Spatially-Weighted Heterogeneity Index | Gaussian distance-decay kernel at multiple bandwidths |
| PTSI | **SL-PTSI** — Spatial-Lag PCA Total Spread Index | PCA on first-difference vectors at lag h |
| JES | **SJCE** — Spatial Join-Count Entropy | Joint entropy of k-nearest-neighbor neighborhoods |
| MVDS | **A-MVDS** — Anisotropic MVDS | Directional variograms + anisotropy ratio |

All four are then size-corrected by log(grid area) and fused into a
**Spatial Master Heterogeneity Index (S-MHI)**.


## 0. Setup & synthetic grid generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.ndimage import gaussian_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

GRID   = 80          # grid is (GRID+1) x (GRID+1) nodes
AREAS  = ['Area A', 'Area B']
PROPS  = ['Structure', 'Porosity', 'Isopach', 'Sw', 'GR']
COLORS = {'Area A': '#185FA5', 'Area B': '#993C1D'}
GEO_WEIGHTS = {'Structure': 0.10, 'Porosity': 0.30,
               'Isopach':   0.20, 'Sw':       0.25, 'GR': 0.15}

# Physical grid spacing (assume 100 m per cell → area = 8 km x 8 km)
CELL_SIZE_M = 100.0
AREA_M2     = (GRID * CELL_SIZE_M) ** 2
print(f"Grid: {GRID+1} x {GRID+1} nodes,  cell = {CELL_SIZE_M} m,  "
      f"total area = {AREA_M2/1e6:.1f} km2")


In [ ]:
# ── Reproduce both area property grids ────────────────────────────────────────
def bf_A(nx, ny):
    return (0.42*np.sin(nx*np.pi*1.2+0.3)*np.cos(ny*np.pi*0.9+0.5)
           +0.28*np.sin(nx*np.pi*2.1+1.1)*np.sin(ny*np.pi*1.7+0.8)
           +0.18*np.cos(nx*np.pi*3.0+0.7)*np.cos(ny*np.pi*2.5+1.3)
           +0.08*np.sin(nx*np.pi*4.0+2.0)*np.cos(ny*np.pi*3.8+0.4)
           +0.14*(1-np.exp(-((nx-0.30)**2+(ny-0.25)**2)/0.015))
           -0.11*np.exp(-((nx-0.72)**2+(ny-0.68)**2)/0.012))

def bf_B(nx, ny):
    return (0.35*np.sin(nx*np.pi*2.5+0.8)*np.cos(ny*np.pi*1.8+1.2)
           +0.22*np.cos(nx*np.pi*4.2+0.4)*np.sin(ny*np.pi*3.1+0.7)
           +0.28*np.sin(nx*np.pi*1.1+1.5)*np.cos(ny*np.pi*2.4+0.3)
           +0.15*np.cos(nx*np.pi*5.5+2.1)*np.cos(ny*np.pi*4.0+1.8)
           +0.18*np.exp(-((nx-0.55)**2+(ny-0.40)**2)/0.018)
           -0.14*np.exp(-((nx-0.22)**2+(ny-0.75)**2)/0.010)
           -0.10*np.exp(-((nx-0.78)**2+(ny-0.20)**2)/0.013))

x = np.linspace(0, 1, GRID+1)
NX, NY = np.meshgrid(x, x)
BFA, BFB = bf_A(NX, NY), bf_B(NX, NY)
C = np.clip

grids = {
    'Area A': {
        'Structure': C(7500+900*np.sin(NX*np.pi*1.2+0.3)*np.cos(NY*np.pi*0.9+0.5)
                       +600*np.sin(NX*np.pi*2.1+1.1)*np.sin(NY*np.pi*1.7+0.8)
                       +400*np.cos(NX*np.pi*3.0+0.7)*np.cos(NY*np.pi*2.5+1.3)
                       +200*np.sin(NX*np.pi*4.0+2.0)*np.cos(NY*np.pi*3.8+0.4)
                       +300*(1-np.exp(-((NX-0.30)**2+(NY-0.25)**2)/0.015))
                       -250*np.exp(-((NX-0.72)**2+(NY-0.68)**2)/0.012), 7500, 10000),
        'Porosity': C(0.075+0.025*BFA, 0.05, 0.10),
        'Isopach':  C(40+20*(-BFA*0.9+0.05*np.sin(NX*5)*np.cos(NY*4)), 20, 60),
        'Sw':       C(0.40-0.18*BFA+0.04*np.cos(NX*6+NY*4), 0.20, 0.60),
        'GR':       C(75+42*(BFA*0.8+0.12*np.sin(NX*7+1)*np.sin(NY*5+0.5)), 30, 120),
    },
    'Area B': {
        'Structure': C(8800+600*np.sin(NX*np.pi*2.5+0.8)*np.cos(NY*np.pi*1.8+1.2)
                       +400*np.cos(NX*np.pi*4.2+0.4)*np.sin(NY*np.pi*3.1+0.7)
                       +300*np.exp(-((NX-0.55)**2+(NY-0.40)**2)/0.018)
                       -200*np.exp(-((NX-0.22)**2+(NY-0.75)**2)/0.010), 7800, 9800),
        'Porosity': C(0.10+0.04*BFB+0.008*np.cos(NX*8)*np.sin(NY*6), 0.06, 0.14),
        'Isopach':  C(40+18*(BFB*0.7+0.15*np.sin(NX*9+1)*np.cos(NY*7+2)), 20, 60),
        'Sw':       C(0.35-0.16*BFB+0.06*np.sin(NX*7+NY*5+0.5)+0.03*np.cos(NX*11), 0.15, 0.55),
        'GR':       C(78+48*(BFB*0.6+0.20*np.sin(NX*10+2)*np.cos(NY*8+1)+0.08*np.sin(NX*15)), 25, 130),
    }
}

# Physical coordinate arrays (metres)
coords_2d = np.stack([NX.ravel() * GRID * CELL_SIZE_M,
                      NY.ravel() * GRID * CELL_SIZE_M], axis=1)  # (N, 2)

# Standardised node x property matrix
X_raw = {a: np.column_stack([grids[a][p].ravel() for p in PROPS]) for a in AREAS}
X_sc  = {a: StandardScaler().fit_transform(X_raw[a]) for a in AREAS}

N_NODES = coords_2d.shape[0]
print(f"Node count: {N_NODES}   Coordinate range: "
      f"0 – {GRID*CELL_SIZE_M:.0f} m in both X and Y")


---
## Index 1 — Spatially-Weighted Heterogeneity Index (SW-WHI)

**Core idea:** Replace the global CV with a *distance-decaying local CV*.
For every node $s$, define a Gaussian kernel weight:

$$w(s, s') = \exp\!\left(-\frac{d(s,s')^2}{2\sigma^2}\right)$$

The kernel-weighted standard deviation and mean at node $s$ are:

$$\mu_\sigma(s) = \frac{\sum_{s'} w(s,s')\, z(s')}{\sum_{s'} w(s,s')}, \quad
  \sigma_\sigma(s) = \sqrt{\frac{\sum_{s'} w(s,s')\,(z(s')-\mu_\sigma)^2}{\sum_{s'} w(s,s')}}$$

The **local spatial CV** is $CV_\sigma(s) = \sigma_\sigma(s)/|\mu_\sigma(s)|$.

SW-WHI is the spatially-averaged, property-weighted mean of $CV_\sigma$:

$$SW\text{-}WHI(\sigma) = \sum_{p} w_p \cdot \overline{CV_{\sigma,p}}$$

Computed at **three bandwidth scales** ($\sigma$ = 200 m, 500 m, 1000 m) to capture
local, intermediate, and regional heterogeneity.


In [ ]:
def gaussian_kernel_cv(grid, coords, sigma_m, subsample=600, seed=0):
    """
    Compute mean kernel-weighted local CV across the grid at bandwidth sigma_m (metres).
    Uses subsampling for speed; full grid used as the neighbour pool.
    """
    rng   = np.random.default_rng(seed)
    N     = coords.shape[0]
    # subsample query nodes
    q_idx = rng.choice(N, size=min(subsample, N), replace=False)
    q_coords = coords[q_idx]            # (Q, 2)
    z_all    = grid.ravel()             # (N,)

    # pairwise distances: (Q, N)
    D = cdist(q_coords, coords)         # metres
    W = np.exp(-D**2 / (2 * sigma_m**2))  # Gaussian weights

    # weighted mean and std
    W_sum  = W.sum(axis=1, keepdims=True) + 1e-12
    mu     = (W * z_all).sum(axis=1) / W_sum[:,0]
    diff2  = (z_all[np.newaxis,:] - mu[:,np.newaxis])**2
    sigma2 = (W * diff2).sum(axis=1) / W_sum[:,0]
    std    = np.sqrt(sigma2)

    local_cv = std / (np.abs(mu) + 1e-12)
    return local_cv.mean(), local_cv   # scalar + per-query array


BANDWIDTHS = [200, 500, 1000]   # metres

sw_whi_results = {area: {} for area in AREAS}   # {area: {sigma: {prop: cv_mean}}}

for area in AREAS:
    for sigma in BANDWIDTHS:
        sw_whi_results[area][sigma] = {}
        for prop in PROPS:
            cv_mean, _ = gaussian_kernel_cv(grids[area][prop], coords_2d,
                                            sigma_m=sigma, subsample=800)
            sw_whi_results[area][sigma][prop] = cv_mean
        # Weighted sum over properties
        whi = sum(GEO_WEIGHTS[p] * sw_whi_results[area][sigma][p] for p in PROPS)
        sw_whi_results[area][sigma]['SW-WHI'] = whi
    print(f"{area}  SW-WHI: "
          + "  ".join(f"sigma={s}m -> {sw_whi_results[area][s]['SW-WHI']:.5f}"
                      for s in BANDWIDTHS))


In [ ]:
# ── SW-WHI multi-scale plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Per-property CV at each sigma
for col, sigma in enumerate(BANDWIDTHS):
    ax = axes[col]
    x_pos = np.arange(len(PROPS)); width = 0.35
    vals_A = [sw_whi_results['Area A'][sigma][p] for p in PROPS]
    vals_B = [sw_whi_results['Area B'][sigma][p] for p in PROPS]
    bA = ax.bar(x_pos-width/2, vals_A, width, label='Area A',
                color=COLORS['Area A'], alpha=0.85)
    bB = ax.bar(x_pos+width/2, vals_B, width, label='Area B',
                color=COLORS['Area B'], alpha=0.85)
    ax.bar_label(bA, fmt='%.4f', padding=2, fontsize=7)
    ax.bar_label(bB, fmt='%.4f', padding=2, fontsize=7)
    ax.set_xticks(x_pos); ax.set_xticklabels(PROPS, rotation=20, ha='right')
    ax.set_ylabel('Kernel-weighted CV')
    ax.set_title(f'sigma = {sigma} m SW-WHI: A={sw_whi_results["Area A"][sigma]["SW-WHI"]:.4f}'
                 f'  B={sw_whi_results["Area B"][sigma]["SW-WHI"]:.4f}',
                 fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Index 1: SW-WHI at three spatial bandwidth scales', fontweight='bold')
plt.tight_layout()
plt.savefig('idx1_SW_WHI.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── SW-WHI heterogeneity spectrum (CV vs sigma) ───────────────────────────────
sigma_range = [100, 200, 350, 500, 750, 1000, 1500, 2000]
spectrum    = {area: [] for area in AREAS}

for area in AREAS:
    for sigma in sigma_range:
        whi = sum(
            GEO_WEIGHTS[p] *
            gaussian_kernel_cv(grids[area][p], coords_2d,
                               sigma_m=sigma, subsample=400)[0]
            for p in PROPS
        )
        spectrum[area].append(whi)
    print(f"{area} spectrum done")

fig, ax = plt.subplots(figsize=(9, 5))
for area in AREAS:
    ax.plot([s/1000 for s in sigma_range], spectrum[area],
            'o-', label=area, color=COLORS[area], linewidth=2.5, markersize=6)
ax.fill_between([s/1000 for s in sigma_range],
                spectrum['Area A'], spectrum['Area B'],
                alpha=0.12, color='gray', label='Difference band')
ax.set_xlabel('Kernel bandwidth sigma (km)')
ax.set_ylabel('SW-WHI')
ax.set_title('SW-WHI heterogeneity spectrum '
             '(left = local patchiness,  right = regional trend)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('idx1b_spectrum.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Spatial map of kernel-weighted local CV (Porosity, sigma=500m) ────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, area in enumerate(AREAS):
    ax = axes[col]
    _, local_cv = gaussian_kernel_cv(grids[area]['Porosity'], coords_2d,
                                     sigma_m=500, subsample=N_NODES)
    cv_map = local_cv.reshape(GRID+1, GRID+1)
    vmax   = np.percentile(cv_map, 97)
    im = ax.imshow(cv_map, origin='lower', cmap='hot_r',
                   vmin=0, vmax=vmax,
                   extent=[0, GRID*CELL_SIZE_M/1000, 0, GRID*CELL_SIZE_M/1000],
                   aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.85, label='Kernel CV (sigma=500m)')
    ax.set_title(f'{area} — Porosity kernel CV map', fontweight='bold',
                 color=COLORS[area])
    ax.set_xlabel('X (km)'); ax.set_ylabel('Y (km)')

plt.suptitle('Index 1 spatial output: Local CV map (Porosity, sigma=500 m)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('idx1c_cv_map.png', dpi=130, bbox_inches='tight')
plt.show()


---
## Index 2 — Spatial-Lag PCA Total Spread Index (SL-PTSI)

**Core idea:** Instead of running PCA on the raw property values $Z(s)$,
run it on the **spatial first-difference vectors**:

$$\Delta Z(s, h) = Z(s + h) - Z(s)$$

where $h$ is a lag vector of distance $|h|$ and azimuth $\theta$.
This means the input to PCA is how much each property *changes* over
a given distance and direction — not the values themselves.

The SL-PTSI at lag $h$ is:

$$SL\text{-}PTSI(h) = 1 - \frac{\lambda_1(h)}{\sum_i \lambda_i(h)}$$

- **Low SL-PTSI at short h**: heterogeneity at short distances is dominated by one property
  (e.g., porosity drives everything locally)
- **High SL-PTSI at short h**: all five properties change independently over short distances
  → complex, multidirectional spatial structure

Computed for multiple lag distances to produce a **spatial complexity spectrum**.


In [ ]:
def spatial_lag_pairs(coords, grid_shape, lag_cells, azimuth_deg=None,
                     tol_cells=1.5, max_pairs=3000, seed=1):
    """
    Find pairs of nodes separated by approximately lag_cells grid cells.
    If azimuth_deg is None, use all directions (isotropic).
    Returns arrays of indices (i, j) for paired nodes.
    """
    rng  = np.random.default_rng(seed)
    rows, cols = grid_shape
    r_idx, c_idx = np.unravel_index(np.arange(rows*cols), grid_shape)

    # For each node sample a random subset as query
    q    = rng.choice(rows*cols, size=min(1500, rows*cols), replace=False)
    q_rc = np.stack([r_idx[q], c_idx[q]], axis=1).astype(float)
    all_rc = np.stack([r_idx, c_idx], axis=1).astype(float)

    D = cdist(q_rc, all_rc)   # (Q, N)

    if azimuth_deg is None:
        # Isotropic: all pairs within [lag-tol, lag+tol]
        mask = (D >= lag_cells - tol_cells) & (D <= lag_cells + tol_cells)
    else:
        # Directional: also constrain azimuth
        az_rad = np.deg2rad(azimuth_deg)
        dR = all_rc[np.newaxis,:,0] - q_rc[:,np.newaxis,0]   # row difference
        dC = all_rc[np.newaxis,:,1] - q_rc[:,np.newaxis,1]
        angle = np.arctan2(dC, dR)
        tol_az = np.deg2rad(22.5)
        angle_ok = np.abs(np.angle(np.exp(1j*(angle - az_rad)))) < tol_az
        mask = ((D >= lag_cells - tol_cells) & (D <= lag_cells + tol_cells) & angle_ok)

    qi, ni = np.where(mask)
    # Map back to original node indices
    src = q[qi]; dst = ni
    # Subsample to max_pairs
    if len(src) > max_pairs:
        sel = rng.choice(len(src), max_pairs, replace=False)
        src, dst = src[sel], dst[sel]
    return src, dst


def sl_ptsi(area_grids, grid_shape, lag_cells_list, azimuth_deg=None):
    """
    Compute SL-PTSI for a list of lag distances.
    Returns list of (lag_m, ptsi) tuples.
    """
    # Build standardised grid stack: (N, 5)
    stack = np.column_stack([StandardScaler().fit_transform(
                area_grids[p].ravel().reshape(-1,1)).ravel()
                for p in PROPS])
    results = []
    for lag in lag_cells_list:
        src, dst = spatial_lag_pairs(None, grid_shape, lag,
                                     azimuth_deg=azimuth_deg, max_pairs=3000)
        if len(src) < 20:
            results.append((lag * CELL_SIZE_M, np.nan)); continue
        delta = stack[dst] - stack[src]    # (n_pairs, 5)
        pca   = PCA(n_components=5)
        pca.fit(delta)
        ptsi  = 1.0 - pca.explained_variance_ratio_[0]
        results.append((lag * CELL_SIZE_M, ptsi))
    return results


LAG_CELLS = [1, 2, 3, 5, 8, 12, 18, 25, 35]
grid_shape = (GRID+1, GRID+1)

ptsi_iso = {}
for area in AREAS:
    ptsi_iso[area] = sl_ptsi(grids[area], grid_shape, LAG_CELLS)
    print(f"{area}: " + "  ".join(f"lag={r[0]:.0f}m->{r[1]:.3f}" for r in ptsi_iso[area]))


In [ ]:
# ── SL-PTSI spatial complexity spectrum ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for area in AREAS:
    lags  = [r[0]/1000 for r in ptsi_iso[area]]
    ptsis = [r[1] for r in ptsi_iso[area]]
    valid = [(l,p) for l,p in zip(lags,ptsis) if not np.isnan(p)]
    lv, pv = zip(*valid)
    ax.plot(lv, pv, 'o-', label=area, color=COLORS[area], linewidth=2.5, markersize=6)

ax.axhline(0.8, color='gray', linestyle=':', linewidth=1,
           label='Max PTSI (fully independent)')
ax.set_xlabel('Lag distance (km)')
ax.set_ylabel('SL-PTSI')
ax.set_title('Spatial complexity spectrum '
             '(high SL-PTSI = properties change independently at that scale)',
             fontweight='bold')
ax.legend(); ax.set_ylim(0, 0.9)

# ── Directional SL-PTSI at a fixed lag (8 cells) ─────────────────────────────
ax2 = axes[1]
azimuths = [0, 45, 90, 135]
az_labels = ['N-S (0°)', 'NE-SW (45°)', 'E-W (90°)', 'NW-SE (135°)']
fixed_lag = 8

for area in AREAS:
    az_ptsi = []
    for az in azimuths:
        res = sl_ptsi(grids[area], grid_shape, [fixed_lag], azimuth_deg=az)
        az_ptsi.append(res[0][1] if not np.isnan(res[0][1]) else 0)
    ax2.plot(azimuths, az_ptsi, 'o-', label=area, color=COLORS[area],
             linewidth=2.5, markersize=8)

ax2.set_xticks(azimuths); ax2.set_xticklabels(az_labels, rotation=15)
ax2.set_ylabel('SL-PTSI')
ax2.set_title(f'Directional SL-PTSI at lag = {fixed_lag*CELL_SIZE_M:.0f} m '
              '(anisotropy = difference between directions)', fontweight='bold')
ax2.legend()

plt.suptitle('Index 2: Spatial-Lag PCA Total Spread Index (SL-PTSI)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('idx2_SL_PTSI.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── SL-PTSI: PC1 loading maps at two scales ───────────────────────────────────
# Shows which property drives spatial change at short vs long range
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for row, lag in enumerate([2, 15]):
    for col, area in enumerate(AREAS):
        ax = axes[row, col]
        stack = np.column_stack([StandardScaler().fit_transform(
                    grids[area][p].ravel().reshape(-1,1)).ravel() for p in PROPS])
        src, dst = spatial_lag_pairs(None, grid_shape, lag, max_pairs=4000)
        delta = stack[dst] - stack[src]
        pca   = PCA(n_components=5).fit(delta)
        loads = np.abs(pca.components_[0])

        bars = ax.bar(PROPS, loads, color=COLORS[area], alpha=0.85)
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
        ax.set_ylim(0, 0.85)
        ptsi_val = 1 - pca.explained_variance_ratio_[0]
        ax.set_title(f'{area}  |  lag = {lag*CELL_SIZE_M:.0f} m '
                     f'SL-PTSI = {ptsi_val:.3f}   PC1 var = '
                     f'{pca.explained_variance_ratio_[0]*100:.1f}%',
                     fontweight='bold', color=COLORS[area])
        ax.set_ylabel('|PC1 loading|')

plt.suptitle('SL-PTSI: Which property drives spatial change at each scale?',
             fontweight='bold')
plt.tight_layout()
plt.savefig('idx2b_loading_maps.png', dpi=130, bbox_inches='tight')
plt.show()


---
## Index 3 — Spatial Join-Count Entropy (SJCE)

**Core idea:** Instead of computing joint entropy across *all* nodes globally
(which ignores space), compute it within a **spatial neighbourhood** of radius $r$
centred on each node, then average over the grid.

For each node $s$, collect its $k$ nearest neighbours within radius $r$.
Discretise all five properties of those neighbours into bins.
Compute the local joint entropy $H_r(s)$ of the neighbourhood.
The SJCE at radius $r$ is:

$$SJCE(r) = \frac{1}{N}\sum_{s} H_r(s) \;/\; H_{max}$$

where $H_{max} = n_{props} \cdot \log_2(n_{bins})$ is the theoretical maximum.

Plotting SJCE as a function of $r$ produces a **spatial entropy spectrum**:
- Rising quickly → heterogeneity builds up rapidly with distance (patchy)
- Rising slowly  → property combinations change gradually (continuous)
- High plateau   → area explores most of the joint state space


In [ ]:
def sjce_at_radius(area_grids, coords, radius_m, n_bins=5,
                   subsample=500, seed=2):
    """
    Compute Spatial Join-Count Entropy at a given neighbourhood radius.
    Returns (sjce_norm, per_node_entropy_array).
    """
    rng    = np.random.default_rng(seed)
    N      = coords.shape[0]
    n_props = len(PROPS)

    # Discretise each property globally into equal-frequency bins
    stack = np.column_stack([area_grids[p].ravel() for p in PROPS])
    binned = np.zeros_like(stack, dtype=int)
    for j in range(n_props):
        q = np.linspace(0, 100, n_bins+1)
        edges = np.percentile(stack[:, j], q)
        edges[0] -= 1e-10; edges[-1] += 1e-10
        binned[:, j] = np.digitize(stack[:, j], edges[1:-1])

    # Subsample query nodes
    q_idx = rng.choice(N, size=min(subsample, N), replace=False)
    H_local = np.zeros(len(q_idx))

    D = cdist(coords[q_idx], coords)   # (Q, N)

    for i, qi in enumerate(q_idx):
        nbrs = np.where(D[i] <= radius_m)[0]
        if len(nbrs) < 3:
            H_local[i] = 0.0; continue
        combos = [tuple(binned[n]) for n in nbrs]
        cnt    = Counter(combos)
        total  = sum(cnt.values())
        p      = np.array(list(cnt.values())) / total
        H_local[i] = -np.sum(p * np.log2(p + 1e-12))

    H_max = n_props * np.log2(n_bins)
    sjce_norm = H_local.mean() / H_max
    return sjce_norm, H_local, q_idx


RADII_M = [150, 300, 500, 800, 1200, 1800, 2500]

sjce_spectrum = {area: [] for area in AREAS}
for area in AREAS:
    for r in RADII_M:
        val, _, _ = sjce_at_radius(grids[area], coords_2d, radius_m=r,
                                   subsample=600)
        sjce_spectrum[area].append(val)
    print(f"{area}: " + "  ".join(f"r={r}m->{sjce_spectrum[area][i]:.3f}"
                                   for i, r in enumerate(RADII_M)))


In [ ]:
# ── SJCE spatial entropy spectrum ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for area in AREAS:
    ax.plot([r/1000 for r in RADII_M], sjce_spectrum[area],
            'o-', label=area, color=COLORS[area], linewidth=2.5, markersize=6)

ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='Theoretical max')
ax.set_xlabel('Neighbourhood radius (km)')
ax.set_ylabel('Normalized SJCE')
ax.set_title('Spatial entropy spectrum '
             '(fast rise = patchy,  slow rise = gradual transition)',
             fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.05)

# Slope of SJCE spectrum (rate of entropy increase with distance)
ax2 = axes[1]
slopes = {}
for area in AREAS:
    r_km  = np.array([r/1000 for r in RADII_M])
    sjce  = np.array(sjce_spectrum[area])
    slope = np.polyfit(r_km, sjce, 1)[0]
    slopes[area] = slope
    ax2.plot(r_km, sjce, 'o-', color=COLORS[area], linewidth=2, label=area)
    ax2.plot(r_km, np.polyval(np.polyfit(r_km, sjce, 1), r_km),
             '--', color=COLORS[area], linewidth=1, alpha=0.6,
             label=f'{area} slope={slope:.3f}')

ax2.set_xlabel('Neighbourhood radius (km)')
ax2.set_ylabel('Normalized SJCE')
ax2.set_title('SJCE with linear trend '
              '(steeper slope = heterogeneity builds faster with distance)',
              fontweight='bold')
ax2.legend(fontsize=8)

plt.suptitle('Index 3: Spatial Join-Count Entropy (SJCE)', fontweight='bold')
plt.tight_layout()
plt.savefig('idx3_SJCE.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Spatial map of per-node entropy (r = 500 m) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, area in enumerate(AREAS):
    ax = axes[col]
    _, H_local, q_idx = sjce_at_radius(grids[area], coords_2d,
                                       radius_m=500, subsample=N_NODES)
    # Map back to grid
    H_max   = len(PROPS) * np.log2(5)
    H_norm  = H_local / H_max
    ent_map = np.full(N_NODES, np.nan)
    ent_map[q_idx] = H_norm
    ent_map = ent_map.reshape(GRID+1, GRID+1)

    im = ax.imshow(ent_map, origin='lower', cmap='inferno',
                   vmin=0, vmax=1,
                   extent=[0, GRID*CELL_SIZE_M/1000, 0, GRID*CELL_SIZE_M/1000],
                   aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.85, label='Local SJCE (r=500m)')
    ax.set_title(f'{area} — per-node SJCE map', fontweight='bold',
                 color=COLORS[area])
    ax.set_xlabel('X (km)'); ax.set_ylabel('Y (km)')
    ax.text(0.02, 0.97, f'mean = {np.nanmean(H_norm):.3f}',
            transform=ax.transAxes, va='top', fontsize=9,
            color='white', fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.4, pad=3))

plt.suptitle('SJCE spatial map: where is joint entropy concentrated? (r=500 m)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('idx3b_sjce_map.png', dpi=130, bbox_inches='tight')
plt.show()


---
## Index 4 — Anisotropic MVDS (A-MVDS)

The original MVDS used an **isotropic** variogram, collapsing all directions.
A-MVDS computes **directional variograms** at four azimuths (0°, 45°, 90°, 135°)
and adds an **anisotropy correction factor**:

$$A = \frac{a_{max}}{a_{min}}$$

where $a_{max}$ and $a_{min}$ are the maximum and minimum fitted variogram ranges
across all directions.  $A = 1$ = perfectly isotropic.  $A > 1$ = elongated
heterogeneity structure (e.g., channel-parallel vs cross-channel).

The corrected score is:

$$A\text{-}MVDS = MVDS_{iso} \times A$$

This penalises areas where heterogeneity is strongly directional, since
cross-dip wells will sample very different conditions from along-dip wells.


In [ ]:
def directional_variogram(grid, n_lags=15, max_lag_frac=0.40,
                         azimuth_deg=0, az_tol=22.5,
                         n_sample=1500, seed=3):
    """Compute directional experimental variogram."""
    flat = grid.ravel()
    rng  = np.random.default_rng(seed)
    idx  = rng.choice(len(flat), size=min(n_sample, len(flat)), replace=False)
    ri, ci = np.unravel_index(idx, grid.shape)
    coords = np.stack([ri, ci], axis=1).astype(float)
    vals   = flat[idx]

    dists  = squareform(pdist(coords))
    max_d  = dists.max() * max_lag_frac
    lags   = np.linspace(0, max_d, n_lags+1)

    # Azimuth mask
    N = len(idx)
    dR = coords[np.newaxis,:,0] - coords[:,np.newaxis,0]
    dC = coords[np.newaxis,:,1] - coords[:,np.newaxis,1]
    angles = np.degrees(np.arctan2(dC, dR)) % 180   # fold to [0,180)
    az_mod = azimuth_deg % 180
    az_ok  = np.abs(angles - az_mod) <= az_tol

    gamma, counts = [], []
    for lo, hi in zip(lags[:-1], lags[1:]):
        dist_ok = (dists > lo) & (dists <= hi)
        mask    = dist_ok & az_ok
        if mask.sum() < 5:
            gamma.append(np.nan); counts.append(0); continue
        ii, jj = np.where(mask)
        gamma.append(0.5 * np.mean((vals[ii]-vals[jj])**2))
        counts.append(mask.sum())

    centers = (lags[:-1]+lags[1:])/2
    return centers, np.array(gamma)


def spherical(h, nugget, sill, rang):
    h = np.asarray(h, float)
    return np.where(h <= rang,
                    nugget+(sill-nugget)*(1.5*(h/rang)-0.5*(h/rang)**3), sill)


AZIMUTHS   = [0, 45, 90, 135]
AZ_LABELS  = ['N-S', 'NE-SW', 'E-W', 'NW-SE']

amvds_results = {area: {} for area in AREAS}

for area in AREAS:
    all_ranges   = {p: [] for p in PROPS}
    all_integrals= {p: [] for p in PROPS}

    for prop in PROPS:
        for az in AZIMUTHS:
            h, gamma = directional_variogram(grids[area][prop], azimuth_deg=az)
            valid = ~np.isnan(gamma)
            if valid.sum() < 4:
                all_ranges[prop].append(np.nan)
                all_integrals[prop].append(np.nan)
                continue
            try:
                p0 = [gamma[valid].min()*0.1, gamma[valid].max(),
                      h[valid].max()*0.5]
                popt, _ = curve_fit(spherical, h[valid], gamma[valid],
                                    p0=p0, maxfev=8000,
                                    bounds=([0,0,1],[np.inf,np.inf,np.inf]))
                rang = popt[2]; sill = popt[1]
                h_fine = np.linspace(0, h[valid].max(), 400)
                gfit   = spherical(h_fine, *popt)
                integ  = np.trapezoid(gfit / (sill+1e-12), h_fine)
                all_ranges[prop].append(rang)
                all_integrals[prop].append(integ)
            except Exception:
                all_ranges[prop].append(np.nan)
                all_integrals[prop].append(np.nan)

    # Anisotropy ratio per property
    aniso = {}
    for prop in PROPS:
        r_arr = [v for v in all_ranges[prop] if not np.isnan(v)]
        aniso[prop] = max(r_arr)/min(r_arr) if len(r_arr) >= 2 and min(r_arr) > 0 else 1.0

    # Isotropic MVDS (mean over azimuths and properties)
    mvds_iso = np.nanmean([v for p in PROPS for v in all_integrals[p]])

    # Anisotropy correction: weighted mean of per-property anisotropy ratios
    A = sum(GEO_WEIGHTS[p]*aniso[p] for p in PROPS)

    amvds_results[area] = dict(
        all_ranges=all_ranges,
        all_integrals=all_integrals,
        aniso=aniso,
        A=A,
        mvds_iso=mvds_iso,
        amvds=mvds_iso * A
    )
    print(f"{area}:  MVDS_iso={mvds_iso:.3f}   A={A:.3f}   A-MVDS={mvds_iso*A:.3f}")
    print(f"  Anisotropy per property: " +
          " ".join(f"{p}={aniso[p]:.2f}" for p in PROPS))


In [ ]:
# ── Rose diagram of variogram ranges (Porosity) ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, area in enumerate(AREAS):
    ax = fig.add_subplot(1, 3, col+1, polar=True)
    r_vals = amvds_results[area]["all_ranges"]["Porosity"]
    angles_rad = [np.deg2rad(az) for az in AZIMUTHS]
    az_full = angles_rad + [a+np.pi for a in angles_rad]
    r_full  = r_vals + r_vals
    az_full.append(az_full[0]); r_full.append(r_full[0])
    valid_mask = [not np.isnan(rv) for rv in r_full]
    az_plot = [a for a,v in zip(az_full, valid_mask) if v]
    r_plot  = [rv for rv,v in zip(r_full, valid_mask) if v]
    ax.plot(az_plot, r_plot, "o-", color=COLORS[area], linewidth=2)
    ax.fill(az_plot, r_plot, alpha=0.2, color=COLORS[area])
    aniso_val = amvds_results[area]["aniso"]["Porosity"]
    title_str = area + " Porosity range rose  A=" + str(round(aniso_val, 2))
    ax.set_title(title_str, fontweight="bold", color=COLORS[area], pad=15)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

ax3 = axes[2]
for i, area in enumerate(AREAS):
    rv = amvds_results[area]
    ax3.bar(i-0.18, rv["mvds_iso"], 0.32, label="Isotropic MVDS",
            color=COLORS[area], alpha=0.5)
    ax3.bar(i+0.18, rv["amvds"], 0.32,
            label="A-MVDS x" + str(round(rv["A"], 2)),
            color=COLORS[area], alpha=0.9)
    ax3.text(i-0.18, rv["mvds_iso"]+0.3, str(round(rv["mvds_iso"], 2)),
             ha="center", fontsize=9, color=COLORS[area])
    ax3.text(i+0.18, rv["amvds"]+0.3, str(round(rv["amvds"], 2)),
             ha="center", fontsize=9, color=COLORS[area], fontweight="bold")

ax3.set_xticks([0, 1]); ax3.set_xticklabels(AREAS)
ax3.set_ylabel("Score")
ax3.set_title("Isotropic MVDS vs A-MVDS (taller = anisotropy amplified)",
              fontweight="bold")
plt.suptitle("Index 4: Anisotropic MVDS (A-MVDS)", fontweight="bold")
plt.tight_layout()
plt.savefig("idx4_AMVDS.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# ── Anisotropy ratio per property ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(PROPS)); width = 0.35

for i, area in enumerate(AREAS):
    a_vals = [amvds_results[area]['aniso'][p] for p in PROPS]
    bars = ax.bar(x_pos+(i-0.5)*width, a_vals, width,
                  label=area, color=COLORS[area], alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1,
           label='Isotropic (A=1)')
ax.set_xticks(x_pos); ax.set_xticklabels(PROPS)
ax.set_ylabel('Anisotropy ratio  a_max / a_min')
ax.set_title('Per-property spatial anisotropy '
             '(>1 = elongated correlation structure, e.g., channel orientation)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('idx4b_anisotropy.png', dpi=130, bbox_inches='tight')
plt.show()


---
## Size correction

All indices should be normalised by the physical size of the study area to ensure
a larger area is not penalised simply for covering more ground.
We apply a **log-area correction**:

$$Index_{corr} = \frac{Index}{\log_{10}(Area_{km^2})}$$

Shannon entropy and spatial variance both scale approximately as $\log(N)$
with sample size $N$, so dividing by $\log(Area)$ removes the size bias.
Both areas here share the same grid dimensions, so the correction is equal
and does not change the ranking — but it becomes critical when comparing
areas of different physical size.


In [ ]:
AREA_KM2    = AREA_M2 / 1e6
log_area    = np.log10(AREA_KM2)
print(f"Study area: {AREA_KM2:.1f} km2   log10(area) = {log_area:.3f}")
print("(Both areas share the same grid, so size correction is equal here.)")
print("If areas differed in size, divide each index by its own log10(area_km2).")
print()

# Select representative scalar values for MHI
# SW-WHI: use sigma=500m
swwhi = {a: sw_whi_results[a][500]['SW-WHI'] / log_area for a in AREAS}

# SL-PTSI: use median across lag spectrum
sl_ptsi_scalar = {}
for area in AREAS:
    ptsi_vals = [r[1] for r in ptsi_iso[area] if not np.isnan(r[1])]
    sl_ptsi_scalar[area] = np.median(ptsi_vals) / log_area

# SJCE: use value at r=500m
sjce_500 = {}
for area in AREAS:
    idx_500 = RADII_M.index(500)
    sjce_500[area] = sjce_spectrum[area][idx_500] / log_area

# A-MVDS
amvds = {a: amvds_results[a]['amvds'] / log_area for a in AREAS}

for area in AREAS:
    print(f"{area}:  SW-WHI={swwhi[area]:.5f}  SL-PTSI={sl_ptsi_scalar[area]:.5f}"
          f"  SJCE={sjce_500[area]:.5f}  A-MVDS={amvds[area]:.5f}")


---
## Spatial Master Heterogeneity Index (S-MHI)

All four size-corrected spatial indices are normalised to [0, 1] and averaged:

$$S\text{-}MHI = \frac{1}{4}\left(\widetilde{SW\text{-}WHI} + \widetilde{SL\text{-}PTSI} + \widetilde{SJCE} + \widetilde{A\text{-}MVDS}\right)$$


In [ ]:
def norm2(d):
    lo, hi = min(d.values()), max(d.values())
    span = hi - lo + 1e-12
    return {k: (v-lo)/span for k, v in d.items()}

sw_n  = norm2(swwhi)
sl_n  = norm2(sl_ptsi_scalar)
sj_n  = norm2(sjce_500)
am_n  = norm2(amvds)

SMHI = {a: np.mean([sw_n[a], sl_n[a], sj_n[a], am_n[a]]) for a in AREAS}

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'SW-WHI (raw)':   {a: round(swwhi[a], 6)           for a in AREAS},
    'SL-PTSI (raw)':  {a: round(sl_ptsi_scalar[a], 6)  for a in AREAS},
    'SJCE (raw)':     {a: round(sjce_500[a], 6)         for a in AREAS},
    'A-MVDS (raw)':   {a: round(amvds[a], 6)            for a in AREAS},
    'SW-WHI (norm)':  {a: round(sw_n[a], 4)             for a in AREAS},
    'SL-PTSI (norm)': {a: round(sl_n[a], 4)             for a in AREAS},
    'SJCE (norm)':    {a: round(sj_n[a], 4)             for a in AREAS},
    'A-MVDS (norm)':  {a: round(am_n[a], 4)             for a in AREAS},
    'S-MHI':          {a: round(SMHI[a], 4)             for a in AREAS},
}).T

print("=" * 62)
print("SPATIAL COMPOSITE LATERAL VARIABILITY SCORECARD")
print("=" * 62)
print(summary.to_string())
winner = max(SMHI, key=SMHI.get)
print(f" S-MHI  ->  Area A: {SMHI['Area A']:.4f}  |  Area B: {SMHI['Area B']:.4f}")
print(f"=> {winner} is overall MORE laterally heterogeneous (spatially).")


In [ ]:
# ── Final master dashboard ────────────────────────────────────────────────────
index_names  = ['SW-WHI', 'SL-PTSI', 'SJCE', 'A-MVDS']
norm_dicts   = [sw_n, sl_n, sj_n, am_n]
raw_dicts    = [swwhi, sl_ptsi_scalar, sjce_500, amvds]
descriptions = [
    'Kernel-weighted CV (sigma=500m)',
    'Spatial-lag PCA spread (median over lags)',
    'Neighbourhood entropy (r=500m)',
    'Directional variogram integral x anisotropy',
]

fig = plt.figure(figsize=(17, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Top row: individual index bars
for col in range(3):
    ax = fig.add_subplot(gs[0, col])
    if col < len(index_names):
        name, norm, raw, desc = (index_names[col], norm_dicts[col],
                                 raw_dicts[col],   descriptions[col])
        vals = [norm[a] for a in AREAS]
        bars = ax.bar(AREAS, vals, color=[COLORS[a] for a in AREAS],
                      alpha=0.85, width=0.45)
        for bar, area in zip(bars, AREAS):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.015,
                    f'norm={norm[area]:.3f} raw={raw[area]:.5f}',
                    ha='center', va='bottom', fontsize=8,
                    color=COLORS[area], fontweight='bold')
        ax.set_ylim(0, 1.35); ax.set_ylabel('Normalized score')
        ax.set_title(f'{name} {desc}', fontweight='bold')
        ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)

# Extra bar for A-MVDS in top-right
ax_am = fig.add_subplot(gs[0, 2])
col = 3
name, norm, raw, desc = (index_names[col], norm_dicts[col],
                         raw_dicts[col],   descriptions[col])
vals = [norm[a] for a in AREAS]
bars = ax_am.bar(AREAS, vals, color=[COLORS[a] for a in AREAS],
                 alpha=0.85, width=0.45)
for bar, area in zip(bars, AREAS):
    ax_am.text(bar.get_x()+bar.get_width()/2,
               bar.get_height()+0.015,
               f'norm={norm[area]:.3f} raw={raw[area]:.5f}',
               ha='center', va='bottom', fontsize=8,
               color=COLORS[area], fontweight='bold')
ax_am.set_ylim(0, 1.35); ax_am.set_ylabel('Normalized score')
ax_am.set_title(f'{name} {desc}', fontweight='bold')
ax_am.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)

# Radar
ax_radar = fig.add_subplot(gs[1, 0], polar=True)
radar_labels = index_names + ['S-MHI']
angles = np.linspace(0, 2*np.pi, len(radar_labels), endpoint=False).tolist()
angles += angles[:1]
smhi_n = norm2(SMHI)
score_sets = {
    'Area A': [sw_n['Area A'], sl_n['Area A'], sj_n['Area A'],
               am_n['Area A'], smhi_n['Area A']],
    'Area B': [sw_n['Area B'], sl_n['Area B'], sj_n['Area B'],
               am_n['Area B'], smhi_n['Area B']],
}
for area, scores in score_sets.items():
    vals = scores + scores[:1]
    ax_radar.plot(angles, vals, 'o-', linewidth=2.5, color=COLORS[area], label=area)
    ax_radar.fill(angles, vals, alpha=0.18, color=COLORS[area])
ax_radar.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontsize=9)
ax_radar.set_ylim(0, 1)
ax_radar.set_title('Spatial indices radar', fontweight='bold', pad=18)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

# S-MHI bar
ax_mhi = fig.add_subplot(gs[1, 1])
mhi_vals = [SMHI[a] for a in AREAS]
bars = ax_mhi.bar(AREAS, mhi_vals,
                  color=[COLORS[a] for a in AREAS], alpha=0.9, width=0.45)
for bar, area in zip(bars, AREAS):
    ax_mhi.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.01,
                f'{SMHI[area]:.4f}',
                ha='center', va='bottom',
                fontsize=14, fontweight='bold', color=COLORS[area])
ax_mhi.set_ylim(0, max(mhi_vals)*1.45)
ax_mhi.set_ylabel('S-MHI score')
ax_mhi.set_title('Spatial Master Heterogeneity Index (S-MHI)', fontweight='bold')
winner = max(SMHI, key=SMHI.get)
ax_mhi.text(0.5, 0.92, f'=> {winner} more heterogeneous',
            transform=ax_mhi.transAxes, ha='center', fontsize=10,
            color=COLORS[winner], fontweight='bold')

# Heatmap
ax_heat = fig.add_subplot(gs[1, 2])
heat_data = np.array([[sw_n[a], sl_n[a], sj_n[a], am_n[a]] for a in AREAS])
im = ax_heat.imshow(heat_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax_heat.set_xticks(range(4)); ax_heat.set_xticklabels(index_names, rotation=15)
ax_heat.set_yticks(range(2)); ax_heat.set_yticklabels(AREAS)
for i in range(2):
    for j in range(4):
        ax_heat.text(j, i, f'{heat_data[i,j]:.3f}', ha='center', va='center',
                     fontsize=11, fontweight='bold',
                     color='white' if heat_data[i,j] < 0.4 else '#1a1a1a')
plt.colorbar(im, ax=ax_heat, shrink=0.8, label='Normalized score')
ax_heat.set_title('Spatial index heatmap', fontweight='bold')

plt.suptitle('Spatial Master Heterogeneity Index (S-MHI) — size-corrected dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('SMHI_master_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()


---
## Interpretation guide

| Index | What spatial feature it captures | Key parameter |
|-------|----------------------------------|--------------|
| **SW-WHI** | How much property values *spread* around each location, weighted by distance | Bandwidth sigma (km) |
| **SL-PTSI** | How independently the 5 properties *change together* at each spatial scale | Lag distance (km) |
| **SJCE** | How many distinct joint property combinations appear in each neighbourhood | Radius r (km) |
| **A-MVDS** | Spatial rate of change + directionality of heterogeneity structure | Variogram range & anisotropy A |
| **S-MHI** | Unified spatial heterogeneity scalar (all four combined, size-corrected) | — |

### Key differences from the non-spatial versions

| Feature | Original (non-spatial) | Spatial upgrade |
|---------|----------------------|-----------------|
| Shuffling invariance | Would give same score if you shuffled all node values | Score changes if values are shuffled — structure matters |
| Scale sensitivity | Single global number | Spectrum across distances — reveals at what scale heterogeneity dominates |
| Directional bias | Ignored | A-MVDS explicitly quantifies anisotropy ratio |
| Physical units | Dimensionless ratio | Tied to coordinate system (metres), size-corrected |
| Well spacing guidance | None | Bandwidth/lag spectra directly map to optimal well spacing |
